In [1]:
%store -r

In [2]:
import json
import os.path

import commute_dm.core
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

In [4]:
def write_interface_to_json_file(interface, output_file_path, description):
    """Serialize a `commute_dm.core.get_interface` result to a JSON file.

    `interface` is `{uniprot_id: [{"collection": <Collection node>,
    "entry": <CollectionEntry node>, "node": <Protein node>}]}`. The join itself
    lives in `commute_dm.core.get_interface`; this notebook only archives it.
    """
    data = []
    for identifier, elements in interface.items():
        data.append(
            {
                "annotation": {"namespace": "uniprot", "identifier": identifier},
                "model_elements": [
                    {
                        "collection": element["collection"]["name"],
                        "entry_file_path": element["entry"]["file_path"],
                        "protein_element_id": element["node"].element_id,
                        "protein_name": element["node"]["name"],
                    }
                    for element in elements
                ],
            }
        )
    output = {"description": description, "data": data}
    with open(output_file_path, "w") as f:
        json.dump(output, f)

In [5]:
COVID_PD_INTERFACE_FILE = os.path.join(INTERFACE_DIR, "covid_pd.json")
COVID_AD_INTERFACE_FILE = os.path.join(INTERFACE_DIR, "covid_ad.json")
COVID_PD_AD_INTERFACE_FILE = os.path.join(INTERFACE_DIR, "covid_pd_ad.json")
COVID_AF_PD_AF_AD_INTERFACE_FILE = os.path.join(INTERFACE_DIR, "covid_af_pd_af_ad.json")
COVID_AF_AD_INTERFACE_FILE = os.path.join(INTERFACE_DIR, "covid_af_ad.json")

We remake the directory where we store the interfaces:

In [6]:
commute_dm.utils.remake_dir(INTERFACE_DIR)

## Computing the interface between the maps based on annotations

### Interface between the COVID DM CD and the PD DM CD

We compute the interface between the COVID and PD maps based on shared UniProt annotations, i.e., we query every UniProt id such that:
- the id is carried by a `:Protein` annotation key (`urn:miriam:uniprot:<id>`) in the `COVID_DM_CD` collection, **and**
- the same id is carried by a `:Protein` in the `PD_DM_CD` collection.

For each such id we collect all the proteins (with their collection and entry) that carry it.

In [7]:
interface = commute_dm.core.get_interface(session, ["COVID_DM_CD", "PD_DM_CD"])
len(interface)

358

We save the interface to a JSON file:

In [8]:
write_interface_to_json_file(
    interface,
    COVID_PD_INTERFACE_FILE,
    (
        "Interface between the COVID maps and the PD maps, based on shared UniProt annotations.\n"
        "This file is generated by the get_interfaces notebook.\n"
        "The interface is formed of all proteins carrying a UniProt annotation that is shared\n"
        "(same UniProt id) between the COVID_DM_CD and PD_DM_CD collections."
    ),
)

### Interface between the COVID DM CD and the AD KG BEL

We save the interface to a JSON file:

### Interface between the COVID DM CD, the PD DM CD, and the AD KG BEL

We save the interface to a JSON file:

### Interface between the COVID DM CD AF, the PD DM CD AF, and the AD KG BEL

We save the interface to a JSON file:

### Interface between the COVID DM CD AF, the PD DM CD AF, and the AD KG CD AF

The three-way pairing `4_10` and `4_20` filter COVID → PD by: every UniProt id carried by a
`:Protein` annotation key in all three of `COVID_DM_CD_AF`, `PD_DM_CD_AF` and `AD_KG_CD_AF`.

`AD_KG_CD_AF` holds only the projected nodes that carry a signed modulation -- `2_10` drops the
rest -- so the identifiers of nodes that could never have seeded a walk are not in it. After that
drop the interface *is* the seedable set.

In [9]:
interface = commute_dm.core.get_interface(
    session, ["COVID_DM_CD_AF", "PD_DM_CD_AF", "AD_KG_CD_AF"]
)
len(interface)

111

In [10]:
write_interface_to_json_file(
    interface,
    COVID_AF_PD_AF_AD_INTERFACE_FILE,
    (
        "Interface between the COVID AF maps, the PD AF maps, and the AD KG stored as a\n"
        "CellDesigner collection (AD_KG_CD_AF), based on shared UniProt annotations.\n"
        "This file is generated by the get_interfaces notebook.\n"
        "The interface is formed of all proteins carrying a UniProt annotation that is shared\n"
        "(same UniProt id) across the COVID_DM_CD_AF, PD_DM_CD_AF and AD_KG_CD_AF collections."
    ),
)

### Interface between the COVID DM CD AF and the AD KG CD AF

The two-way pairing the COVID x AD comorbidity sub-maps of `4_10` run on once the AD side is a
stored CellDesigner collection.

In [11]:
interface = commute_dm.core.get_interface(session, ["COVID_DM_CD_AF", "AD_KG_CD_AF"])
len(interface)

159

In [12]:
write_interface_to_json_file(
    interface,
    COVID_AF_AD_INTERFACE_FILE,
    (
        "Interface between the COVID AF maps and the AD KG stored as a CellDesigner\n"
        "collection (AD_KG_CD_AF), based on shared UniProt annotations.\n"
        "This file is generated by the get_interfaces notebook.\n"
        "The interface is formed of all proteins carrying a UniProt annotation that is shared\n"
        "(same UniProt id) between the COVID_DM_CD_AF and AD_KG_CD_AF collections.\n"
        "This is the interface the COVID x AD (CD) comorbidity sub-maps (4_10) are built around."
    ),
)